# 2024 Presidential Election Results Analysis and Plots

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 16,
        "figure.figsize": (10, 7.5),
        "figure.dpi": 200,
        "figure.edgecolor": "k",
        "figure.facecolor": "w",
    }
)

In [ ]:
# Load 2024 data
try:
    df_24 = pd.read_feather("2024.ft")
    print("Loaded 2024 data successfully")
    print(f"Data shape: {df_24.shape}")
    print(f"Columns: {list(df_24.columns)}")
except FileNotFoundError:
    print("2024.ft not found. Please run the data processing first.")
    df_24 = None

In [ ]:
if df_24 is not None:
    # Basic summary statistics
    print("\n2024 Presidential Election Summary:")
    print("=" * 50)
    
    # Total votes
    total_votes = df_24["pres_24_total"].sum()
    print(f"Total votes cast: {total_votes:,}")
    
    # Candidate totals
    candidate_cols = [col for col in df_24.columns if "pres_24_" in col and "pct" not in col and "total" not in col]
    for col in candidate_cols:
        candidate_name = col.replace("pres_24_", "")
        total_votes_candidate = df_24[col].sum()
        percentage = (total_votes_candidate / total_votes) * 100
        print(f"{candidate_name}: {total_votes_candidate:,} votes ({percentage:.1f}%)")
    
    # County-level analysis
    print(f"\nCounty-level analysis:")
    print(f"Number of counties: {len(df_24)}")
    
    # Find counties with highest turnout
    df_24_sorted = df_24.sort_values("pres_24_total", ascending=False)
    print(f"\nTop 5 counties by turnout:")
    for i, row in df_24_sorted.head().iterrows():
        print(f"{row['county']}: {row['pres_24_total']:,} votes")
    
    # Find counties with highest Trump support
    trump_col = [col for col in df_24.columns if "pres_24_trump" in col and "pct" in col][0]
    df_24_sorted_trump = df_24.sort_values(trump_col, ascending=False)
    print(f"\nTop 5 counties by Trump support:")
    for i, row in df_24_sorted_trump.head().iterrows():
        print(f"{row['county']}: {row[trump_col]*100:.1f}%")
    
    # Find counties with highest Harris support
    harris_col = [col for col in df_24.columns if "pres_24_harris" in col and "pct" in col][0]
    df_24_sorted_harris = df_24.sort_values(harris_col, ascending=False)
    print(f"\nTop 5 counties by Harris support:")
    print(f"\nTop 5 counties by Harris support:")
    for i, row in df_24_sorted_harris.head().iterrows():
        print(f"{row['county']}: {row[harris_col]*100:.1f}%")
    
    # Create visualizations
    plt.figure(figsize=(15, 10))
    
    # Plot 1: Vote distribution by county
    plt.subplot(2, 2, 1)
    plt.hist(df_24["pres_24_total"], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    plt.xlabel("Total Votes per County")
    print(f"\nTop 5 counties by Harris support:")
    plt.ylabel("Number of Counties")
    plt.title("Distribution of Total Votes by County")
    plt.grid(True, alpha=0.3)
    
    # Plot 2: Trump vs Harris support scatter
    plt.subplot(2, 2, 2)
    plt.scatter(df_24[trump_col], df_24[harris_col], alpha=0.6, color='red')
    plt.xlabel("Trump Support (%)")
    plt.ylabel("Harris Support (%)")
    plt.title("Trump vs Harris Support by County")
    plt.grid(True, alpha=0.3)
    
    # Add diagonal line for reference
    max_val = max(df_24[trump_col].max(), df_24[harris_col].max())
    plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.5)
    
    # Plot 3: County turnout vs Trump support
    plt.subplot(2, 2, 3)
    plt.scatter(df_24["pres_24_total"], df_24[trump_col], alpha=0.6, color='red')
    plt.xlabel("Total Votes per County")
    plt.ylabel("Trump Support (%)")
    plt.title("County Turnout vs Trump Support")
    plt.grid(True, alpha=0.3)
    
    # Plot 4: County turnout vs Harris support
    plt.subplot(2, 2, 4)
    plt.scatter(df_24["pres_24_total"], df_24[harris_col], alpha=0.6, color='blue')
    plt.xlabel("Total Votes per County")
    plt.ylabel("Harris Support (%)")
    plt.title("County Turnout vs Harris Support")
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Correlation analysis
    print(f"\nCorrelation Analysis:")
    print("=" * 30)
    
    # Calculate correlations between key variables
    key_cols = ["pres_24_total", trump_col, harris_col]
    correlation_matrix = df_24[key_cols].corr()
    print(correlation_matrix)
    
    # Summary statistics by region (if we have geographic data)
    print(f"\nRegional Analysis:")
    print("=" * 30)
    
    # Group counties by size (small, medium, large)
    df_24["county_size"] = pd.cut(df_24["pres_24_total"], 
                                  bins=3, 
                                  labels=["Small", "Medium", "Large"])
    
    size_analysis = df_24.groupby("county_size").agg({
        trump_col: ["mean", "std"],
        harris_col: ["mean", "std"],
        "pres_24_total": ["mean", "std"]
    }).round(3)
    
    print(size_analysis)
    
    # Save analysis results
    analysis_results = {
        "total_votes": total_votes,
        "counties_analyzed": len(df_24),
        "trump_total": df_24[trump_col.replace("_pct", "")].sum(),
        "harris_total": df_24[harris_col.replace("_pct", "")].sum(),
        "correlation_trump_harris": df_24[trump_col].corr(df_24[harris_col])
    }
    
    print(f"\nAnalysis Summary:")
    print("=" * 30)
    for key, value in analysis_results.items():
        if isinstance(value, float):
            print(f"{key}: {value:.3f}")
        else:
            print(f"{key}: {value:,}")